In [ ]:
from uuid import uuid4
import pandas as pd
from datetime import datetime

# Read the CSV file
nw_data = pd.read_csv("../Input/NEYctDNA.csv")

nw_data['PlacerOrderNumber'] = nw_data['PlacerOrderNumber'].fillna('').astype(str)

nw_data['PostCode'] = nw_data['PostCode'].fillna('')
nw_data['HospitalSpellIdentifier'] = nw_data['HospitalSpellIdentifier'].fillna('').astype(str).replace('nan', '')

for index, row in nw_data.iterrows():
    #print(row)

    MSH = 'MSH|^~\\&|IGENE|MFT|EPIC|MFT|'+datetime.today().strftime('%Y%m%d%H%M%S')+'||ORU^R01|'+str(uuid4())+'|T|2.3'
    print(MSH)
    PID = 'PID||'+str(row['NHSNumber']) +'|'+str(row['HospitalNumber']) +'^^^'+row['RequestingOrganisationCode'] +'^MR||'+row['PatientFamilyName'] +'^'+row['PatientGivenName'] +'||'+row['DateOfBirth'].replace('-','') +'|'+row['AdministrativeSex'][0] +'|||^^^^'+row['PostCode']
    print(PID)

    # PV1-19 (Visit Number) carries the Hospital Spell Identifier; omit the
    # segment entirely when a spell hasn't been assigned, as in the source data.
    hospitalSpellIdentifier = row['HospitalSpellIdentifier'].strip()
    PV1 = None
    if hospitalSpellIdentifier:
        pv1Fields = [''] * 19
        pv1Fields[1] = 'U'
        pv1Fields[18] = hospitalSpellIdentifier
        PV1 = 'PV1|' + '|'.join(pv1Fields)
        print(PV1)

    ORC = 'ORC|RE|'+str(row['PlacerOrderNumber']) +'|||||||||||||||||||'+row['RequestingOrganisationName'] +'^^'+row['RequestingOrganisationCode'] +'^^^ODS'
    print(ORC)
    OBR = 'OBR|1|'+str(row['PlacerOrderNumber']) +'|'+str(row['TestAccessionIdentifier']) +'|'+row['TestCode'] +'^'+row['NGTDTestName'] +'^IGEAP||||||||||202510060000||^^^^^^^^PROVID^^^^PROVID||||||20251014155916|||F|||||||SH_JJE^Edgerley^Jonathan'
    print(OBR)
    OBX = 'OBX|1|CE|'+row['TestCode'] +'^'+row['NGTDTestName'] +'^IGENE|PDF|^IGene^application/pdf^Base64^JVBERi0xLjQKMSAwIG9iago8PC9UeXBlIC9DYXRhbG9nCi9QYWdlcyAyIDAgUgo+PgplbmRvYmoK MiAwIG9iago8PC9UeXBlIC9QYWdlcwovS2lkcyBbMyAwIFJdCi9Db3VudCAxCj4+CmVuZG9iagozIDAgb2JqCjw8L1R5cGUgL1BhZ2UKL1BhcmVudCAyIDAgUgovTWVkaWFCb3ggWzAgMCA1OTUgODQy XQovQ29udGVudHMgNSAwIFIKL1Jlc291cmNlcyA8PC9Qcm9jU2V0IFsvUERGIC9UZXh0XQovRm9udCA8PC9GMSA0IDAgUj4+Cj4+Cj4+CmVuZG9iago0IDAgb2JqCjw8L1R5cGUgL0ZvbnQKL1N1YnR5 cGUgL1R5cGUxCi9OYW1lIC9GMQovQmFzZUZvbnQgL0hlbHZldGljYQovRW5jb2RpbmcgL01hY1JvbWFuRW5jb2RpbmcKPj4KZW5kb2JqCjUgMCBvYmoKPDwvTGVuZ3RoIDUzCj4+CnN0cmVhbQpCVAov RjEgMjAgVGYKMjIwIDQwMCBUZAooRHVtbXkgUERGKSBUagpFVAplbmRzdHJlYW0KZW5kb2JqCnhyZWYKMCA2CjAwMDAwMDAwMDAgNjU1MzUgZgowMDAwMDAwMDA5IDAwMDAwIG4KMDAwMDAwMDA2MyAw MDAwMCBuCjAwMDAwMDAxMjQgMDAwMDAgbgowMDAwMDAwMjc3IDAwMDAwIG4KMDAwMDAwMDM5MiAwMDAwMCBuCnRyYWlsZXIKPDwvU2l6ZSA2Ci9Sb290IDEgMCBSCj4+CnN0YXJ0eHJlZgo0OTUKJSVFT0YK|||N|||F'
    print(OBX)
    NTE='NTE|1|L|'+row['NGTDTestCode'] +'='+row['NGTDTestName']
    print(NTE)
    segments = [MSH, PID]
    if PV1:
        segments.append(PV1)
    segments += [ORC, OBR, OBX, NTE]
    with open('Input/V2/R01/ctdna' + str(row['NHSNumber'])+'.txt', 'w') as v2:
        v2.write('\r'.join(segments))
        v2.close()